In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

Load Project Utilities & Initialize Notebook Widgets

In [0]:
%run /Workspace/Users/naveengowda7796@gmail.com/consolidated_pipeline/setup/Utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

bronze silver gold


In [0]:
#create Databricks widgets for catalog and datasource for easier identification and modification 
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-nm/{data_source}/*.csv'
print(base_path)

s3://sportsbar-nm/customers/*.csv


**Bronze**

In [0]:
df = (
    spark.read.format("csv")
                .option("header", True)
                .option("inferSchema", True)
                .load(base_path)
                .withColumn("read_timestamp", F.current_timestamp())
                .select("*", "_metadata.file_name","_metadata.file_size")
)                
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-04-19T09:25:17.995Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-04-19T09:25:17.995Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-04-19T09:25:17.995Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-04-19T09:25:17.995Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-04-19T09:25:17.995Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-04-19T09:25:17.995Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-04-19T09:25:17.995Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-04-19T09:25:17.995Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-04-19T09:25:17.995Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-04-19T09:25:17.995Z,customers.csv,1404


In [0]:
#check data type
df.printSchema()


root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

**Silver Processing**

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)
                      

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-04-19 09:25:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-04-19 09:25:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-04-19 09:25:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-04-19 09:25:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-04-19 09:25:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-04-19 09:25:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-04-19 09:25:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-04-19 09:25:...|customers.csv|     1404|
|     7891

In [0]:
df_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



**Transformations**

In [0]:

df_duplicates = df_bronze.groupBy("customer_id").count().where("count >1")
display(df_duplicates)

customer_id,count
789321,2
789503,2
789522,2
789603,2


In [0]:
#drop duplicates
print('Rows before duplicates dropped: ',df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print('Rows after duplicates dropped: ',df_silver.count())



Rows before duplicates dropped:  39
Rows after duplicates dropped:  35


In [0]:
# display values with leading spaces
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-04-19T09:25:33.758Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-04-19T09:25:33.758Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-04-19T09:25:33.758Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404


In [0]:
# remove those trim values
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
# display values with leading spaces
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,file_name,file_size


In [0]:
# Data Quality Fix: Correcting City Typos
df_silver.select('city').distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [0]:
#typo dictionary
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
    }

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver.replace(city_mapping, subset = ["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(),None)
        .when(F.col("city").isin(allowed), F.col("city"))
        .otherwise(None)
    )
)

#sanity check
df_silver.select('city').distinct().show()

+---------+
|     city|
+---------+
|New Delhi|
|Hyderabad|
|Bengaluru|
+---------+



In [0]:
# title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(),None)
    .otherwise(F.initcap("customer_name"))
)

df_silver.select('customer_name').distinct().show()


+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



In [0]:
# Handling missing cities
df_silver.filter(F.col("city").isNull()).show(truncate = False)

+-----------+-------------------+----+--------------------------+-------------+---------+
|customer_id|customer_name      |city|read_timestamp            |file_name    |file_size|
+-----------+-------------------+----+--------------------------+-------------+---------+
|789403     |Sprintx Nutrition  |NULL|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789603     |Recovery Lane      |NULL|2026-04-19 09:25:33.758066|customers.csv|1404     |
+-----------+-------------------+----+--------------------------+-------------+---------+



In [0]:
null_customer_names = ["Sprintx Nutrition", "Zenathlete Foods","Primefuel Nutrition", "Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate = False)


+-----------+-------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timestamp            |file_name    |file_size|
+-----------+-------------------+---------+--------------------------+-------------+---------+
|789401     |Sprintx Nutrition  |Bengaluru|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789402     |Sprintx Nutrition  |Hyderabad|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789403     |Sprintx Nutrition  |NULL     |2026-04-19 09:25:33.758066|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL     |2026-04-19 09:25:33.758066|customers.csv|1404     |
|789421     |Zenathlete Foods   |Hyderabad|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789422     |Zenathlete Foods   |New Delhi|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789520     |Primefuel Nutrition|Bengaluru|2026-04-19 09:25:33.758066|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL     |2026-04

In [0]:
#Business confirmation note: City corrections confirmed by business team
customer_city_fix = {
    #Sprintx Nutrition
    789403: "New Delhi",

   #Zenathlete Foods 
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    #Recovery Lane 
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)
display(df_fix)


customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id","left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")  #replace null with fixed city
    )
    .drop("fixed_city")
)

display(df_silver)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789101,Endurance Foods,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-04-19T09:25:33.758Z,customers.csv,1404
789121,Hydroboost Nutrition,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404
789122,Hydroboost Nutrition,New Delhi,2026-04-19T09:25:33.758Z,customers.csv,1404
789201,Fitfuel Market,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404
789202,Fitfuel Market,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404
789203,Fitfuel Market,New Delhi,2026-04-19T09:25:33.758Z,customers.csv,1404
789220,Macrobite Superfoods,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404
789221,Macrobite Superfoods,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404


In [0]:
# Convert customer_id to string
df_silver =df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


In [0]:
#Standardizing Customer Attributes
df_silver = (
    df_silver
    # build final customer column: "Customer-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-","customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )

    #static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-04-19T09:25:33.758Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-04-19T09:25:33.758Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-04-19T09:25:33.758Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition


In [0]:
df_silver.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

Gold processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

#take required columns only
#customer_id, customer_name, city, read_timestamp,file_name,file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")
display(df_gold.limit(10))

customer_id,customer_name,city,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bengaluru,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market,Hyderabad,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods,Hyderabad,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
df_gold.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.{data_source}")

In [0]:
#Merging Data source
delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)


In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()